# Transfer to NAS (round-robin drives)

Run this notebook **during acquisition**, alongside HAL/Dave, when using
`round_robin_drives` mode (`prepare_imaging/03`'s `DATA_DRIVES`).

Unlike `02_round_scheduler.ipynb`'s transfer step -- which only moves a round
once its mosaics have already been built **locally** (`tracker.is_round_done`)
-- this notebook transfers a round's raw data (plus the small `round_info.csv`
/ `round_bit_color_map.csv` / `positions_*.txt` / `settings/*.xml` files a
cluster-side `ExperimentMetadata.load()` needs alongside it) as soon as HAL
has **finished writing it**, with no dependency on any local QC analysis.
This is the intended companion to moving QC analysis off the microscope
computer entirely and onto a SLURM cluster instead (see
`07_cluster_submit_analysis.ipynb`) -- `01_fov_scheduler.ipynb`/
`02_round_scheduler.ipynb` don't need to run at all in that workflow.

Each tick:
- Syncs the small metadata/positions/settings files to `TRANSFER_DEST`.
- For every round that is **fully written** (every expected raw file exists
  on disk) and **not on the drive HAL is actively writing right now**, starts
  a background transfer of that round's data directory to `TRANSFER_DEST`.
- Marks each round transferred with a zero-byte sentinel once its copy
  succeeds (retried automatically on the next tick if it fails).

Requires `ANALYSIS_MODE = "round_robin_drives"` (the "which drive is hot right
now" signal this relies on) and a real `TRANSFER_DEST`.

## 1 — Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/analysis/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config   import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.progress        import ProgressTracker
from MERci.scheduler       import TransferScheduler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

Edit the cells below to match your experiment. `TRANSFER_DEST` is required
(unlike `01`/`02`, where it's optional) -- this notebook exists only to
transfer.

In [ ]:
SAMPLE_NAME = SAMPLE_DIR.name

# ── Image file format (must match what HAL writes) ───────────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── Transfer destination (required) ─────────────────────────────
TRANSFER_DEST = None   # e.g. r"\\NAS\experiments\LT027"

if TRANSFER_DEST is None:
    raise ValueError("Set TRANSFER_DEST to the NAS destination root before running this notebook.")

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Image suffix : {IMAGE_SUFFIX}")
print(f"Transfer dest: {TRANSFER_DEST}")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
    analysis_mode  = "round_robin_drives",
    transfer_dest  = TRANSFER_DEST,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

print(f"Rounds       : {meta.n_rounds}")
print(f"FOVs         : {meta.n_fovs}")
print(f"Data roots   : {config.all_data_roots}")
print(f"Transfer dest: {config.transfer_dest}")

## 3 — Check current progress

Run this cell at any time to see how many rounds are fully written vs.
already transferred (independent of any local QC — no `.fov_done`/
`.round_done` sentinel is required here).

In [ ]:
round_ids     = meta.valid_round_ids()
n_written     = sum(1 for r in round_ids if meta.round_fully_written(r))
n_transferred = sum(1 for r in round_ids if tracker.is_round_transferred(r))
print(f"Rounds fully written : {n_written} / {len(round_ids)}")
print(f"Rounds transferred   : {n_transferred} / {len(round_ids)}")

## 4 — Transfer scheduler

**Run this cell and leave it running** during acquisition.

Interrupt the kernel (`■` button) to stop the loop; any in-flight transfer
thread is a daemon thread and will simply be abandoned (the round stays
"not transferred" and is retried the next time this notebook runs).

In [ ]:
def show_tick(tick):
    from IPython.display import clear_output
    clear_output(wait=True)
    active_rid = meta.actively_writing_round()
    active_drv = meta.drive_of_round(active_rid) if active_rid is not None else None
    print(f"[tick {tick['iteration']}] started {tick['transfers_started']} transfer(s) this tick.")
    print(f"Active round / drive: {active_rid} / {active_drv or 'none'}")
    n_written     = sum(1 for r in meta.valid_round_ids() if meta.round_fully_written(r))
    n_transferred = sum(1 for r in meta.valid_round_ids() if tracker.is_round_transferred(r))
    print(f"Rounds written / transferred: {n_written} / {n_transferred}  (of {meta.n_rounds})")

TransferScheduler(config, meta, tracker).run_loop(on_tick=show_tick)